# Faruq-v3 — Geometry Family-Effect Decomposition

Posthoc descriptive decomposition of the completed three-seed GEO1 vs GEO-C0 confirmation. **No training, no inference, no locked test.** This notebook does not create a new confirmation gate; it only measures whether the already-confirmed aggregate GEO effect differs systematically across size-defined object families.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import importlib, json, os, shutil, subprocess, sys, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/circle-cpe-screening'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone = ['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(3):
    result = subprocess.run(clone)
    if result.returncode == 0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt == 2: raise RuntimeError('Git clone gagal tiga kali')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO/'src'))
importlib.invalidate_caches()
os.chdir(REPO)
print('COMMIT:', subprocess.check_output(['git','rev-parse','HEAD'], cwd=REPO, text=True).strip())


In [ ]:
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

SOURCE_REL = 'experiments/faruq-v3-geometry-conditioning-paired-confirmation-v1/val_reports/geometry_conditioning_paired_three_seed_confirmation.json'
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(SOURCE_REL,))
SOURCE = require_project_artifact(PROJECT_ROOT, SOURCE_REL)
OUTPUT_ROOT = PROJECT_ROOT/'experiments/faruq-v3-geometry-family-effect-decomposition-v1'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT = OUTPUT_ROOT/'geometry_family_effect_decomposition.json'
print('SOURCE:', SOURCE)
print('OUTPUT:', OUTPUT)


In [ ]:
command = [sys.executable,'-m','pytest','-q','tests/test_geometry_family_effect_decomposition.py']
print('FOCUSED TEST:', ' '.join(command), flush=True)
subprocess.run(command, cwd=REPO, check=True)


In [ ]:
command = [
    sys.executable, '-u', '-m', 'coffee_detector.analysis.geometry_family_effect_decomposition',
    '--source', str(SOURCE), '--output', str(OUTPUT),
]
print('AUDIT:', ' '.join(command), flush=True)
subprocess.run(command, cwd=REPO, check=True)
assert OUTPUT.is_file(), OUTPUT


In [ ]:
import pandas as pd
from IPython.display import display

result = json.loads(OUTPUT.read_text(encoding='utf-8'))
assert result['evaluation_split'] == 'val'
assert result['test_images_accessed'] is False and result['test_opened'] is False
assert result['analysis_status'] == 'posthoc_descriptive_decomposition_no_new_gate'

family_rows = []
for family, values in result['aggregate_families'].items():
    family_rows.append({
        'family': family,
        'mean_delta': values['mean_delta'],
        'min_delta': values['min_delta'],
        'max_delta': values['max_delta'],
        'positive_seeds': values['positive_seeds'],
        'negative_seeds': values['negative_seeds'],
        'pattern': values['pattern'],
    })
print('AGGREGATE FAMILY EFFECT — GEO1 minus GEO-C0')
display(pd.DataFrame(family_rows).style.format({'mean_delta':'{:+.2%}','min_delta':'{:+.2%}','max_delta':'{:+.2%}'}))

seed_rows = []
for seed, record in result['per_seed'].items():
    for family, values in record['families'].items():
        seed_rows.append({
            'seed': int(seed), 'family': family,
            'mean_delta': values['mean_delta'],
            'positive_classes': values['positive_classes'],
            'negative_classes': values['negative_classes'],
            'within_family_range': values['within_family_range'],
        })
print('PER-SEED FAMILY EFFECT')
display(pd.DataFrame(seed_rows).style.format({'mean_delta':'{:+.2%}','within_family_range':'{:.2%}'}))

class_rows = []
for name, values in result['aggregate_classes'].items():
    class_rows.append({
        'family': values['family'], 'class': name,
        'mean_delta': values['mean_delta'],
        'positive_seeds': values['positive_seeds'],
        'negative_seeds': values['negative_seeds'],
        'pattern': values['pattern'],
    })
print('CLASS-LEVEL CONSISTENCY')
display(pd.DataFrame(class_rows).sort_values('mean_delta', ascending=False).style.format({'mean_delta':'{:+.2%}'}))

contrast_rows = []
for name, values in result['family_contrasts'].items():
    contrast_rows.append({
        'contrast': name, 'mean_delta': values['mean_delta'],
        'positive_seeds': values['positive_seeds'],
        'negative_seeds': values['negative_seeds'],
        'pattern': values['pattern'],
    })
print('BETWEEN-FAMILY CONTRASTS')
display(pd.DataFrame(contrast_rows).style.format({'mean_delta':'{:+.2%}'}))
print('STATUS:', result['analysis_status'])
print('NEXT:', result['next_action'])
print('OUTPUT:', OUTPUT)
print('Tidak ada training/inference/test pada notebook ini.')
